# 04 · Two-Stage: Retrieval → Ranking

Production pattern:

```
request → two-tower retrieval (top-N) → LightGBM re-ranker → top-K
```

We keep **standalone two-tower** and compare it to **two-stage**. If two-stage does not beat retrieval-alone, that is a useful finding — the ranker features may not yet be rich enough.

In [ ]:
import os
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import pandas as pd

from src.recsys.config import settings
from src.recsys.data import load_dataset
from src.recsys.eval import evaluate, temporal_split
from src.recsys.models import TwoStageRecommender, TwoTowerRecommender

ds = load_dataset()
train, test_positives = temporal_split(ds.interactions)
test_users = list(test_positives.keys())
max_k = max(settings.eval_ks)
print(ds.summary())

## Fit both models

Standalone two-tower vs two-stage (same two-tower architecture as the retriever, then LightGBM re-rank of top-100).

In [ ]:
tt = TwoTowerRecommender(dim=64, epochs=15)
two_stage = TwoStageRecommender(
    retriever=TwoTowerRecommender(dim=64, epochs=15),
    candidate_n=100,
    verbose=True,
)

print("fitting two_tower...")
tt.fit(train)
print("fitting two_stage...")
two_stage.fit(train)

In [ ]:
results = {}
for name, model in [("two_tower", tt), ("two_stage", two_stage)]:
    recs = model.recommend(test_users, k=max_k)
    results[name] = evaluate(recs, test_positives, ks=settings.eval_ks, n_items=ds.n_items)

results_df = pd.DataFrame(results).T
cols = [c for c in results_df.columns if c.endswith(f"@{settings.top_k}")]
results_df[cols]

In [ ]:
ax = results_df[[f"recall@{settings.top_k}", f"ndcg@{settings.top_k}", f"hitrate@{settings.top_k}"]].plot.bar(
    figsize=(8, 4), rot=0, title=f"Two-tower alone vs two-stage @ K={settings.top_k}"
)
ax.set_ylabel("score"); ax.figure.tight_layout()

## How to read the result

- **two_tower better** → retrieval order is already strong; re-ranker features (popularity, avg rating, …) may not add much on this synthetic data.
- **two_stage better** → the ranker is fixing retrieval mistakes using side features.

On real Yelp (richer features later: text, social, geo) the gap often widens in favor of two-stage.

**Next:** social recommender (`models/social.py`), then richer ranker features.